In [ ]:
import pandas as pd

df = pd.read_parquet("../data/processed/fannie_2017_loan_level.parquet")

print(df.shape)
print(df["default_flag"].mean())               # your base rate — memorize this number
print(df.groupby("orig_quarter")["default_flag"].agg(["mean", "size"]))
print(df["LOAN_ID"].is_unique)                  # confirm the reduction did what it should

/Users/ryanquinlan/Downloads/loan-portfolio-optimization/eda
(2046851, 115)
0.034143178961243394
                  mean    size
orig_quarter                  
2017Q1        0.031214  487789
2017Q2        0.032269  492517
2017Q3        0.034398  546663
2017Q4        0.038399  519882
True


**Base rate: 3.4%** default across ~2.05M loans (2017 originations, all four quarters, stable 3.1%–3.8% by quarter). This is the reference point where all "lift" values below are subgroup rate ÷ base rate.

In [ ]:
#check nulls and range
num_cols = ["CSCORE_B", "DTI", "ORIG_RATE", "OLTV", "OCLTV", "ORIG_UPB", "ORIG_TERM"]
for c in num_cols:
    coerced = pd.to_numeric(df[c], errors="coerce")
    print(f"{c:12s} nulls after coerce: {coerced.isna().mean():.3%}  range: {coerced.min()}–{coerced.max()}")
    df[c] = coerced

CSCORE_B     nulls after coerce: 0.077%  range: 445.0–850.0
DTI          nulls after coerce: 0.017%  range: 1.0–63.0
ORIG_RATE    nulls after coerce: 0.000%  range: 1.79–6.125
OLTV         nulls after coerce: 0.000%  range: 2–97
OCLTV        nulls after coerce: 0.000%  range: 2–114
ORIG_UPB     nulls after coerce: 0.000%  range: 5000.0–1223000.0
ORIG_TERM    nulls after coerce: 0.000%  range: 36–360


Data quality: all origination features coerce cleanly to numeric (<0.1% nulls); no Fannie sentinel values (9999 FICO, 999 DTI) contaminating the columns. Ranges are all plausible. Safe to model on.


In [19]:
def rate_by_bin(df, col, bins):
    g = df.groupby(pd.cut(df[col], bins))["default_flag"]
    out = g.agg(["mean", "size"])
    out["lift"] = out["mean"] / df["default_flag"].mean()
    return out

rate_by_bin(df, "CSCORE_B", [300, 620, 660, 700, 740, 780, 850])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_25911/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
CSCORE_B,,,
"(300, 620]",0.123989,1484,3.631449
"(620, 660]",0.108010,102222,3.163443
"(660, 700]",0.073385,254412,2.149328
"(700, 740]",0.043869,422215,1.284844
"(740, 780]",0.023800,579244,0.697064
"(780, 850]",0.011140,685701,0.326285


In [21]:
df.groupby("PURPOSE")["default_flag"].agg(["mean", "size"]).sort_values("mean")
rate_by_bin(df, "DTI", [0, 20, 30, 36, 43, 50, 65])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_25911/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
DTI,,,
"(0, 20]",0.011856,167425,0.347245
"(20, 30]",0.019133,484301,0.560368
"(30, 36]",0.029545,418380,0.865324
"(36, 43]",0.042624,601751,1.248388
"(43, 50]",0.054968,374633,1.609940
"(50, 65]",0.000000,21,0.000000


In [22]:
rate_by_bin(df, "OLTV", [0, 60, 70, 80, 90, 95, 100])

/var/folders/5g/nyyrx9tx5nbfgnrym8rw723m0000gn/T/ipykernel_25911/3838274286.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(pd.cut(df[col], bins))["default_flag"]


,mean,size,lift
OLTV,,,
"(0, 60]",0.020946,391141,0.613487
"(60, 70]",0.030740,252021,0.900312
"(70, 80]",0.031312,786434,0.917086
"(80, 90]",0.035643,233451,1.043941
"(90, 95]",0.051220,272568,1.500160
"(95, 100]",0.063280,111236,1.853368


In [23]:
df.groupby("STATE")["default_flag"].agg(["mean", "size"]).sort_values("mean", ascending=False).head(15)

,mean,size
STATE,,
VI,0.125000,128
PR,0.081258,2449
FL,0.056946,141082
NY,0.056179,62924
HI,0.053410,6759
LA,0.050447,20041
DC,0.048153,4901
TX,0.046143,164662
NJ,0.045474,47698


Risk hierarchy so far, strongest to weakest:

| Feature   | Pattern                          | Approx. swing | Notes |
|-----------|----------------------------------|---------------|-------|
| CSCORE_B  | Smooth, monotonic decline        | ~10×          | Dominant signal. <620 defaults 12.4% (3.6× lift); 780+ only 1.1% (0.33×). But <660 is a thin slice of the book — most loans are 740+. |
| DTI       | Smooth, monotonic rise           | ~4–5×         | No cliff at the 43 conforming limit; risk rises steadily with leverage. Independent of FICO, so additive. |
| OLTV      | Flat through 80, sharp tail rise | ~3×           | Signal is in the tail: ~3% up to 80% LTV, jumps to 5.1% (90–95) and 6.3% (95–100). Captures equity/skin-in-the-game. |
| PURPOSE   | Weak ordering                    | ~1.5×         | Cash-out refi (C) 3.8% > purchase (P) 3.5% > rate-term refi (R) 2.6%. Real but minor next to the above. |

Caveats for the team:
- **ORIG_RATE deliberately not treated as a predictor.** Rate is priced from the same risk assessed at origination, so using it to predict default is partly circular / leakage-adjacent. Documented as a relationship, not a feature. Revisit before modeling.
- With ~2M rows, every subgroup difference is "statistically significant" where we rely on **lift / effect size**, not p-values, to judge what matters.
- Always check the `size` column before trusting a rate (see the DTI 50+ bin: 0% default on only 21 loans = noise, not a finding).